# Fine-tune VieNeu-TTS v3 Turbo trên Colab / Kaggle

Notebook dành cho **một giọng**, dùng trực tiếp các script trong `finetune/` của repo. Chạy các ô từ trên xuống. Mọi lệnh dài đều chạy trong tiến trình riêng, hiển thị log và dừng khi có lỗi.

Dataset đang dùng: `metadata.csv` + `raw_audio/`, gồm **329 clip (~15,28 phút)**. Notebook không đánh giá chất lượng dataset. Bước chuẩn bị của SDK vẫn áp dụng giới hạn thời lượng theo pipeline.

Kết quả: **LoRA adapter**, **model đã merge**, **giọng dựng sẵn** và file ZIP tải về. Notebook không tự tải model lên Hugging Face.

## Trước khi chạy

- **Colab:** mở notebook này, chọn runtime có **GPU**, rồi kết nối. Có thể upload `dataset.zip` hoặc đọc từ Google Drive. Bật `SAVE_TO_DRIVE=True` ở ô cấu hình nếu muốn lưu checkpoint trực tiếp vào Drive.
- **Kaggle:** import notebook này, bật **GPU** và **Internet**, thêm dataset qua phần Input/Add Input. Dữ liệu trong `/kaggle/input` chỉ đọc; notebook chép vào `/kaggle/working`. Nếu được cấp hai GPU, script chỉ dùng **một GPU**.
- Cần Internet để clone repo, cài thư viện và tải model/codec. GPU và ổ đĩa trống phải đủ cho model, cache và ZIP; chuẩn bị vài GB trống. Thời gian thực tế phụ thuộc GPU/CPU được cấp.

### Đưa dataset từ máy Windows lên

Chạy PowerShell tại thư mục repo trên máy của bạn:

```powershell
Compress-Archive -Path .\finetune\dataset\metadata.csv,.\finetune\dataset\raw_audio -DestinationPath .\dataset.zip
```

Nếu `dataset.zip` đã tồn tại, đổi tên file đích hoặc thêm `-Force` để ghi đè. Upload ZIP này lên Colab, Google Drive hoặc tạo Kaggle Dataset. Cấu trúc ZIP có thể có một thư mục bọc ngoài:

```text
dataset.zip
  metadata.csv
  raw_audio/
    clip_000001.wav
    ...
```

Trên Kaggle, dataset đã giải nén có cấu trúc như trên cũng được. Không cần upload cả repo hay cài môi trường trên Windows.


## 1. Cấu hình

Chỉ cần sửa các biến trong ô dưới. `DATA_MODE` có ba lựa chọn:

| Giá trị | Cách sử dụng |
|---|---|
| `colab_upload` | Colab mở hộp chọn ZIP ở bước 4. |
| `drive` | Điền `DATA_SOURCE` là đường dẫn ZIP hoặc thư mục trên Drive. |
| `kaggle` | Tự tìm dataset trong `/kaggle/input`, không cần nhập đường dẫn. |

`REPO_REVISION` cố định phiên bản code được dùng để soạn notebook. Đổi thành `main` nếu chủ động muốn dùng phiên bản mới hơn. `RUN_NAME` nên khác nhau cho mỗi lần thử, tránh trộn log/checkpoint. `OUTPUT_ROOT` để rỗng sẽ chọn thư mục mặc định theo nền tảng.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, hashlib, csv
from datetime import datetime, timezone

IS_KAGGLE = Path('/kaggle/working').is_dir()
IS_COLAB = not IS_KAGGLE and 'google.colab' in sys.modules
if not IS_KAGGLE and not IS_COLAB:
    try:
        import google.colab
        IS_COLAB = True
    except ImportError:
        raise RuntimeError('Notebook này cần chạy trong Colab hoặc Kaggle (Linux, GPU CUDA).')

REPO_URL = 'https://github.com/pnnbao97/VieNeu-TTS.git'
REPO_REVISION = 'bd0f88eb8de7d560753a6172ba9c868f309a101c'
BASE_MODEL = 'pnnbao-ump/VieNeu-TTS-v3-Turbo'
MODEL_SUBFOLDER = 'update'

DATA_MODE = 'kaggle' if IS_KAGGLE else 'colab_upload'
DATA_SOURCE = ''  # Kaggle tự tìm; chỉ cần điền khi dùng Drive.
DATASET_CHOICE = None  # Kaggle: chỉ chọn số thứ tự nếu có nhiều dataset phù hợp.
SAVE_TO_DRIVE = False  # Chỉ áp dụng trên Colab
OUTPUT_ROOT = ''  # Ví dụ: /content/drive/MyDrive/VieNeu/output
RUN_NAME = 'my_voice_01'
REBUILD_PARQUET = False  # Đổi True khi đổi audio/text hoặc phiên bản codec/SDK

EPOCHS = 3
MAX_STEPS = 0  # 0 = dùng EPOCHS; số dương = giới hạn số bước optimizer
BATCH_SIZE = 2
GRAD_ACCUM = 8  # Batch hiệu dụng = 2 * 8 = 16
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGET = 'backbone'  # 'all' học thêm acoustic decoder, cần nhiều VRAM/data hơn
MAX_LENGTH = 1024
EVAL_RATIO = 0.05
EVAL_EVERY = 10
SAVE_EVERY = 10
GRAD_CHECKPOINT = False
FORCE_FP32 = False  # Notebook tự bật FP32 nếu GPU không hỗ trợ BF16
SEED = 42

VOICE_NAME = 'Giong cua toi'
VOICE_DESCRIPTION = 'Giọng đã fine-tune từ dataset cá nhân'
VOICE_GENDER = ''  # 'male', 'female', hoặc để rỗng
REFERENCE_FILE = ''  # Đường dẫn tương đối trong raw_audio; rỗng = tự chọn clip 3–8 giây
TEST_TEXT = 'Xin chào, đây là giọng nói của tôi sau khi được tinh chỉnh.'
RUN_DEMO = True
EXPORT_PREPARED_DATA = True  # ZIP thêm train.parquet để tái sử dụng lần sau
DOWNLOAD_ON_COLAB = False  # Đổi True để tự mở tải ZIP sau khi xuất

if DATA_MODE not in {'colab_upload', 'drive', 'kaggle'}:
    raise ValueError('DATA_MODE không hợp lệ.')
if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {'', '.', '..'}:
    raise ValueError('RUN_NAME phải là tên thư mục đơn, không chứa đường dẫn.')
WORK_ROOT = Path('/kaggle/working/vieneu_finetune' if IS_KAGGLE else '/content/vieneu_finetune')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_ROOT / 'repo'
DATASET_DIR = WORK_ROOT / 'dataset'
VENV_DIR = WORK_ROOT / 'venv'
PYTHON = VENV_DIR / 'bin/python'
print('Nền tảng:', 'Kaggle' if IS_KAGGLE else 'Colab')
print('Thư mục làm việc:', WORK_ROOT)


## 2. Kết nối Drive và xác định nơi lưu

Trên Colab, ô này chỉ mount Drive khi dùng `DATA_MODE='drive'`, `SAVE_TO_DRIVE=True`, hoặc đường dẫn output trỏ vào Drive. Đăng nhập khi Colab yêu cầu. Trên Kaggle, kết quả nên nằm trong `/kaggle/working` để xuất thành notebook outputs.

**Ổ đĩa runtime có thể mất khi phiên làm việc kết thúc.** Drive giữ các file đã ghi; trên Kaggle cần lưu một phiên bản notebook kèm outputs và tải file kết quả trước khi bỏ phiên làm việc. Notebook không thể tự bảo đảm dữ liệu tồn tại nếu runtime bị thu hồi giữa lúc ghi file.


In [ ]:
if IS_COLAB and (DATA_MODE == 'drive' or SAVE_TO_DRIVE or OUTPUT_ROOT.startswith('/content/drive/')):
    from google.colab import drive
    drive.mount('/content/drive')
if DATA_MODE == 'drive' and not IS_COLAB:
    raise ValueError('Chế độ drive chỉ dùng trên Colab; Kaggle dùng DATA_MODE=kaggle.')
if DATA_MODE == 'colab_upload' and not IS_COLAB:
    raise ValueError('Upload tương tác này chỉ dùng trên Colab.')
if not OUTPUT_ROOT:
    OUTPUT_ROOT = '/content/drive/MyDrive/VieNeu/output' if IS_COLAB and SAVE_TO_DRIVE else str(WORK_ROOT / 'output')
OUTPUT_DIR = Path(OUTPUT_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_DIR / RUN_NAME
ARTIFACT_DIR = Path('/kaggle/working/vieneu_exports' if IS_KAGGLE else '/content/vieneu_exports')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def run_command(args, *, log_path=None, cwd=None):
    args = [str(x) for x in args]
    print('Chạy:', ' '.join(args), flush=True)
    log = open(log_path, 'w', encoding='utf-8') if log_path else None
    try:
        with subprocess.Popen(args, cwd=str(cwd or WORK_ROOT), env=os.environ.copy(),
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as proc:
            try:
                for line in proc.stdout:
                    print(line, end='', flush=True)
                    if log:
                        log.write(line)
                        log.flush()
                status = proc.wait()
            except BaseException:
                proc.terminate()
                try:
                    proc.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
                raise
        if status:
            raise subprocess.CalledProcessError(status, args)
    finally:
        if log:
            log.close()

def ensure_lora_ready():
    # Kiểm tra bằng chính interpreter chạy train; kernel có thể nhìn thấy gói khác.
    check = "import importlib.util; print(int(importlib.util.find_spec('torchao') is not None))"
    found = subprocess.check_output([str(PYTHON), '-c', check], text=True).strip().splitlines()[-1]
    if found == '1':
        print('Phát hiện torchao trong Python train; gỡ backend tùy chọn trước khi tiếp tục.')
        run_command([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
        run_command([PYTHON, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    preflight = """import importlib.util, torch
spec = importlib.util.find_spec('torchao')
assert spec is None, f'torchao vẫn còn tại {spec.origin}; xem log gỡ gói phía trên.'
from peft import LoraConfig, get_peft_model
small = get_peft_model(torch.nn.Sequential(torch.nn.Linear(4, 4)),
                       LoraConfig(r=2, lora_alpha=4, target_modules=['0']))
small(torch.ones(1, 4)).sum().backward()
assert any(p.grad is not None for name, p in small.named_parameters() if 'lora_' in name)
print('OK: Python train không còn torchao; gắn LoRA và backward thành công.')
"""
    run_command([PYTHON, '-c', preflight])

print('Output:', OUTPUT_DIR)
print('ZIP xuất tại:', ARTIFACT_DIR)


## 3. Lấy code và cài thư viện

Notebook tạo môi trường Python riêng nhưng dùng lại PyTorch CUDA có sẵn của Colab/Kaggle. Môi trường được tạo với `--without-pip`, sau đó pip của kernel cài pip vào môi trường đó qua `--python`, tránh lỗi `ensurepip` trên Kaggle. Các script chạy bằng Python trong môi trường đó, vì vậy không cần đổi kernel. Không cài `vieneu[cuda]` vì extra này ghim một phiên bản torch riêng; notebook dùng `.[finetune]` theo repo.

Ô này có thể mất vài phút. Nếu clone thất bại, kiểm tra Internet và `REPO_URL`/`REPO_REVISION`. Nếu đã đổi revision sau khi chạy ô này, dùng một runtime mới để cài lại nhất quán.

LoRA trong notebook không dùng lượng tử hóa TorchAO. Ô cài đặt gỡ gói tùy chọn `torchao` khỏi runtime và venv để tránh PEFT phát hiện phiên bản cũ có sẵn trên Kaggle/Colab. PyTorch CUDA vẫn được dùng lại. Sau đó ô thử gắn LoRA và backward trên model nhỏ trước khi tải model thật.


In [ ]:
if not (REPO_DIR / '.git').is_dir():
    run_command(['git', 'clone', REPO_URL, REPO_DIR])
run_command(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_REVISION])
run_command(['git', '-C', REPO_DIR, 'checkout', '--detach', 'FETCH_HEAD'])
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
# Kaggle có thể thiếu ensurepip: tạo venv không bootstrap pip.
# Chạy cả khi venv cũ bị tạo dở ở lần chạy trước.
run_command([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', VENV_DIR])
# pip của kernel cài pip vào đúng venv, không cần ensurepip.
run_command([sys.executable, '-m', 'pip', '--python', PYTHON,
             'install', '--upgrade', '--ignore-installed', 'pip'])
run_command([PYTHON, '-m', 'pip', 'install', '-e', str(REPO_DIR) + '[finetune]'])

# LoRA thường không dùng torchao. PEFT vẫn dò backend này và có thể lỗi
# vì torchao cũ được kế thừa từ Kaggle/Colab. Gỡ riêng gói tùy chọn này
# khỏi runtime và venv; giữ nguyên torch CUDA và các dữ liệu đã chuẩn bị.
run_command([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
run_command([PYTHON, '-m', 'pip', 'uninstall', '-y', 'torchao'])

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = str(WORK_ROOT / 'hf_cache')
# Không tải model hay giữ tensor GPU trong kernel notebook.
gpu_probe = """import json, torch
from transformers import Qwen3Config, Qwen3Model
import peft, pyarrow, soundfile, vieneu
from vieneu._v3_turbo_engine.modeling_v3_turbo import VieNeuV3TurboForTTS
# Thực sự gắn LoRA và backward trên model nhỏ để kiểm tra các backend PEFT.
from peft import LoraConfig, get_peft_model
small = get_peft_model(torch.nn.Sequential(torch.nn.Linear(4, 4)),
                       LoraConfig(r=2, lora_alpha=4, target_modules=['0']))
small(torch.ones(1, 4)).sum().backward()
assert any(p.grad is not None for name, p in small.named_parameters() if 'lora_' in name)
del small
assert torch.cuda.is_available(), 'Chưa có GPU CUDA: bật GPU rồi kết nối runtime mới.'
print(json.dumps({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0),
                  'vram_gb': round(torch.cuda.get_device_properties(0).total_memory/2**30, 1),
                  'bf16': torch.cuda.is_bf16_supported(including_emulation=False)}))
"""
probe = subprocess.check_output([str(PYTHON), '-c', gpu_probe], text=True)
GPU_INFO = json.loads(probe.strip().splitlines()[-1])
USE_BF16 = GPU_INFO['bf16'] and not FORCE_FP32
print('Code commit:', CODE_COMMIT)
print('GPU:', GPU_INFO)
print('Kiểu tính toán khi train:', 'BF16 autocast' if USE_BF16 else 'FP32 (--no-bf16)')


## 4. Nạp dataset

**Colab upload:** chọn đúng một `dataset.zip` khi hộp thoại xuất hiện. Nếu ZIP đã nằm trên Drive, dùng `DATA_MODE='drive'` và `DATA_SOURCE` để tránh upload lại.

**Kaggle:** thêm dataset vào Input rồi chạy ô này. Notebook tự tìm thư mục có `metadata.csv` và `raw_audio`, kể cả bên trong các thư mục lồng nhau; nếu chưa giải nén thì tìm ZIP có cấu trúc dataset. Không cần nhập đường dẫn. Đường dẫn cũ không tồn tại sẽ được bỏ qua để tự tìm lại. Nếu có nhiều dataset phù hợp, notebook liệt kê số thứ tự; đặt `DATASET_CHOICE` bằng số tương ứng trong cấu hình để chọn, tránh train nhầm dữ liệu.

Ô này chỉ giải nén/tìm cấu trúc và chép dữ liệu vào thư mục ghi được, không đánh giá chất lượng audio hay transcript. Nếu thay dataset trong cùng runtime, đặt `REBUILD_PARQUET=True`.


In [ ]:
def find_kaggle_dataset(input_root, configured_source='', choice=None):
    """Tìm dataset theo cấu trúc thư mục/ZIP, không dựa vào tên hiển thị Kaggle."""
    input_root = Path(input_root)
    if configured_source:
        configured = Path(configured_source)
        if configured.is_dir() and (configured / 'metadata.csv').is_file() and (configured / 'raw_audio').is_dir():
            return configured
        if configured.is_file() and zipfile.is_zipfile(configured):
            return configured
        print('Đường dẫn cũ không phù hợp; tự tìm trong:', input_root)
    candidates = sorted({p.parent for p in input_root.rglob('metadata.csv')
                         if (p.parent / 'raw_audio').is_dir()}, key=str)
    # Ưu tiên thư mục đã giải nén; chỉ tìm ZIP khi chưa có thư mục phù hợp.
    if not candidates:
        for path in sorted(input_root.rglob('*.zip')):
            try:
                with zipfile.ZipFile(path) as archive:
                    names = [name.replace('\\', '/') for name in archive.namelist()]
                    for name in names:
                        if name.split('/')[-1] == 'metadata.csv':
                            prefix = name[:-len('metadata.csv')] + 'raw_audio/'
                            if any(item.startswith(prefix) and not item.endswith('/') for item in names):
                                candidates.append(path)
                                break
            except (OSError, zipfile.BadZipFile):
                continue
    if not candidates:
        raise FileNotFoundError('Chưa tìm thấy dataset trong /kaggle/input. Thêm dataset vào Input rồi chạy lại ô này.')
    for index, candidate in enumerate(candidates):
        print(f'Dataset [{index}]: {candidate}')
    if choice is not None:
        if isinstance(choice, bool) or not isinstance(choice, int) or not 0 <= choice < len(candidates):
            raise ValueError('DATASET_CHOICE phải là số thứ tự trong danh sách phía trên.')
        selected = candidates[choice]
    elif len(candidates) == 1:
        selected = candidates[0]
    else:
        raise ValueError('Có nhiều dataset phù hợp. Đặt DATASET_CHOICE bằng số thứ tự dataset muốn dùng trong ô cấu hình rồi chạy lại. Không cần nhập đường dẫn.')
    print('Tự chọn dataset:', selected)
    return selected

source = None
if DATA_MODE == 'colab_upload':
    from google.colab import files
    upload_dir = WORK_ROOT / 'uploads'
    upload_dir.mkdir(exist_ok=True)
    previous_cwd = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    if len(uploaded) != 1:
        raise ValueError('Hãy chọn đúng một file dataset.zip.')
    source = upload_dir / next(iter(uploaded))
    del uploaded  # Giải phóng bản ZIP đang giữ trong RAM
elif DATA_MODE == 'kaggle':
    source = find_kaggle_dataset('/kaggle/input', DATA_SOURCE, DATASET_CHOICE)
elif DATA_SOURCE:
    source = Path(DATA_SOURCE)
else:
    raise ValueError('Điền DATA_SOURCE trỏ tới ZIP hoặc thư mục dataset.')
if not source.exists():
    raise FileNotFoundError(source)

if source.is_file():
    if not zipfile.is_zipfile(source):
        raise ValueError('File đầu vào phải là ZIP.')
    with source.open('rb') as f:
        digest = hashlib.file_digest(f, 'sha256').hexdigest() if hasattr(hashlib, 'file_digest') else None
    if digest is None:
        h = hashlib.sha256()
        with source.open('rb') as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b''):
                h.update(chunk)
        digest = h.hexdigest()
    extracted = WORK_ROOT / ('import_' + digest[:12])
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        for entry in archive.infolist():
            target = (extracted / entry.filename).resolve()
            if not target.is_relative_to(extracted.resolve()) or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('ZIP chứa đường dẫn hoặc liên kết không hợp lệ: ' + entry.filename)
        archive.extractall(extracted)
    search_root = extracted
else:
    search_root = source
roots = [search_root] if (search_root / 'metadata.csv').is_file() and (search_root / 'raw_audio').is_dir() else [
    p.parent for p in search_root.rglob('metadata.csv') if (p.parent / 'raw_audio').is_dir()
]
if len(roots) != 1:
    raise ValueError('Cần đúng một thư mục metadata.csv + raw_audio; tìm thấy: ' + repr(roots))
source_root = roots[0]
DATASET_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(source_root / 'metadata.csv', DATASET_DIR / 'metadata.csv')
shutil.copytree(source_root / 'raw_audio', DATASET_DIR / 'raw_audio', dirs_exist_ok=True)
print('Dataset đã sẵn sàng tại:', DATASET_DIR)


## 5. Chuẩn bị `train.parquet`

Script chạy trên **CPU**, tải ONNX codec/speaker encoder, phiên âm text và tạo codes + speaker embedding. GPU có thể chưa hoạt động ở bước này, đó là bình thường. Lần đầu phải tải các tài nguyên model nên có thể chờ lâu trước dòng tiến độ đầu tiên.

Nếu đã chạy thành công trong cùng runtime, ô dùng lại parquet. Nếu có parquet từ phiên trước, có thể chép nó vào đường dẫn in bên dưới để bỏ qua mã hoá lại, miễn là được tạo từ cùng dataset và pipeline tương thích. Đổi `REBUILD_PARQUET=True` khi thay dữ liệu. Kết quả chuẩn bị có thể xuất ZIP ở bước cuối để dùng lại.

**Bản sửa issue #198:** mã đuôi `482` là im lặng hợp lệ; artifact encoder đã biết là `455`. Ô dưới tự sửa guard nhầm của revision được ghim trước khi chuẩn bị dữ liệu. Nếu lần trước dừng với thông báo `codebook-0 = 482`, chạy lại ô này bằng notebook đã cập nhật; không cần sửa audio hoặc tắt kiểm tra artifact.


In [ ]:
# Revision ghim trong notebook có guard cũ: nhầm mã im lặng 482 là pad frame.
# Chỉ sửa guard đã biết; giữ kiểm tra artifact 455 của issue #198.
prepare_script = REPO_DIR / 'finetune/prepare_dataset.py'
prepare_source = prepare_script.read_text(encoding='utf-8')
old_guard = '    code, n_same = last.most_common(1)[0]\n    if len(recs) >= 20 and n_same / len(recs) > 0.5:'
fixed_guard = '    code, n_same = last.most_common(1)[0]\n    if code == 455 and len(recs) >= 20 and n_same / len(recs) > 0.5:'
if old_guard in prepare_source:
    if prepare_source.count(old_guard) != 1:
        raise RuntimeError('Guard prepare_dataset.py không khớp phiên bản dự kiến.')
    prepare_script.write_text(prepare_source.replace(old_guard, fixed_guard), encoding='utf-8')
    print('Đã sửa guard issue #198: chỉ chặn artifact 455, cho phép mã im lặng 482.')

PARQUET_PATH = DATASET_DIR / 'train.parquet'
if REBUILD_PARQUET or not PARQUET_PATH.is_file():
    run_command([PYTHON, '-u', REPO_DIR / 'finetune/prepare_dataset.py',
                 '--dataset-dir', DATASET_DIR, '--base', BASE_MODEL,
                 '--min-sec', '1', '--max-sec', '20'],
                log_path=WORK_ROOT / 'prepare.log')
else:
    print('Dùng lại:', PARQUET_PATH, '(đặt REBUILD_PARQUET=True nếu dữ liệu đã thay đổi)')
if not PARQUET_PATH.is_file():
    raise FileNotFoundError('Script chưa tạo train.parquet; xem log phía trên.')
print('Parquet:', PARQUET_PATH)


## 6. Train LoRA

Mặc định: rank 16, LR `2e-4`, 3 epoch, batch hiệu dụng 16, giữ 5% dữ liệu cho eval. Với 329 dòng hợp lệ, khoảng **313 dòng train / 16 dòng eval**, khoảng **60 bước optimizer** cho 3 epoch theo cách script tính. Số dòng/bước chính xác được in trong log sau khi dựng dataset.

Eval mỗi 10 bước và lưu adapter mỗi 10 bước để có dữ liệu theo dõi trong lượt train ngắn. `loss` và `audio` thường giảm dần; so sánh thêm `eval_loss`. Script **không tự early stopping hoặc chọn checkpoint tốt nhất**. Nếu eval xấu đi, có thể dùng checkpoint trước đó ở bước merge. Log `acc_cb0` là chỉ số theo dõi của pipeline, không phải điểm chất lượng nghe.

**Chạy lại ô train sẽ bắt đầu một lượt train mới.** Nếu thư mục run đã có dữ liệu, notebook tự thêm hậu tố vào tên lượt mới và cập nhật đường dẫn cho các bước merge/xuất kết quả; kết quả cũ được giữ nguyên. Script hiện chưa có `--resume`: checkpoint chỉ lưu adapter, không lưu đủ optimizer/scheduler để tiếp tục chính xác từ chỗ ngắt. Nếu runtime mất, vẫn có thể merge checkpoint đã lưu.


In [ ]:
ensure_lora_ready()  # Xử lý/kiểm tra torchao trước khi tải model và tạo lượt mới.

# Mỗi lần chạy ô train tạo một lượt riêng nếu tên đã có dữ liệu.
# Cập nhật cả RUN_NAME và RUN_DIR để merge/export dùng đúng lượt mới.
requested_name = RUN_NAME
RUN_DIR = OUTPUT_DIR / requested_name
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    suffix = 2
    while (OUTPUT_DIR / f'{requested_name}_{suffix:02d}').exists():
        suffix += 1
    RUN_NAME = f'{requested_name}_{suffix:02d}'
    RUN_DIR = OUTPUT_DIR / RUN_NAME
    print(f'Lượt {requested_name} đã tồn tại. Tự tạo lượt mới: {RUN_NAME}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output lượt train này:', RUN_DIR)
train_args = [PYTHON, '-u', REPO_DIR / 'finetune/train_lora.py',
              '--data', PARQUET_PATH, '--base', BASE_MODEL, '--subfolder', MODEL_SUBFOLDER,
              '--output-dir', OUTPUT_DIR, '--run', RUN_NAME,
              '--epochs', EPOCHS, '--max-steps', MAX_STEPS,
              '--batch-size', BATCH_SIZE, '--grad-accum', GRAD_ACCUM,
              '--lr', LEARNING_RATE, '--r', LORA_R, '--alpha', LORA_ALPHA,
              '--target', LORA_TARGET, '--max-length', MAX_LENGTH,
              '--eval-ratio', EVAL_RATIO, '--eval-every', EVAL_EVERY,
              '--save-every', SAVE_EVERY, '--log-every', 5,
              '--num-workers', 0, '--seed', SEED]
if not USE_BF16:
    train_args.append('--no-bf16')
if GRAD_CHECKPOINT:
    train_args.append('--grad-checkpoint')
# Merge ở tiến trình riêng bên dưới để trả VRAM sau train.
run_command(train_args, log_path=RUN_DIR / 'train_console.log')
print('Adapter cuối:', RUN_DIR / 'adapter')


## 7. Đọc log và chọn adapter

Bảng này đọc file log, không cần tải model vào GPU. `eval_loss` chỉ xuất hiện ở các bước eval định kỳ; eval cuối được in trong `train_console.log`. Giữ `ADAPTER_TO_MERGE=''` để dùng adapter cuối. Muốn chọn checkpoint, điền ví dụ `str(RUN_DIR / 'checkpoint-20')` trong ô merge tiếp theo. Giá trị eval thấp nhất là gợi ý lựa chọn, cần nghe model sau khi merge để quyết định.


In [ ]:
log_file = RUN_DIR / 'train_log.jsonl'
if log_file.is_file():
    records = [json.loads(line) for line in log_file.read_text().splitlines() if line.strip()]
    try:
        import pandas as pd
        display(pd.DataFrame(records))
    except ImportError:
        for record in records:
            print(record)
    evals = [r for r in records if 'eval_loss' in r]
    if evals:
        best = min(evals, key=lambda r: r['eval_loss'])
        print('Eval định kỳ thấp nhất:', best)
        print('Checkpoint tại bước này chỉ có nếu đã được lưu và bước đó chưa phải bước cuối.')
else:
    print('Chưa có train_log.jsonl. Nếu train bị ngắt, xem train_console.log và các checkpoint hiện có.')
print('Các adapter đã lưu:', [p.name for p in sorted(RUN_DIR.glob('*')) if p.is_dir()])


## 8. Merge adapter vào model gốc

Bước này tạo model đầy đủ. Adapter nhỏ cần model gốc để merge; thư mục `merged/` dùng để suy luận với SDK. Nếu đã có adapter từ phiên trước, chép thư mục adapter lên runtime và điền đường dẫn vào `ADAPTER_TO_MERGE`.

Model merge chạy bằng backend **PyTorch/GPU**. SDK hiện chưa tự export model đã fine-tune thành ONNX; không chọn backend ONNX để nghe model này. Codec dùng khi suy luận vẫn có thể được tải riêng từ Hugging Face.


In [ ]:
ADAPTER_TO_MERGE = ''  # Ví dụ: str(RUN_DIR / 'checkpoint-20')
ADAPTER_DIR = Path(ADAPTER_TO_MERGE) if ADAPTER_TO_MERGE else RUN_DIR / 'adapter'
if not (ADAPTER_DIR / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'Không thấy adapter_config.json trong {ADAPTER_DIR}. Chọn checkpoint đã lưu nếu train bị ngắt.')
MERGED_DIR = RUN_DIR / 'merged'
run_command([PYTHON, '-u', REPO_DIR / 'finetune/merge_lora.py',
             '--adapter', ADAPTER_DIR, '--base', BASE_MODEL, '--subfolder', MODEL_SUBFOLDER,
             '--out', MERGED_DIR], log_path=RUN_DIR / 'merge.log')
print('Model merge:', MERGED_DIR)


## 9. Đóng gói giọng

Dùng một clip của chính người nói trong dataset để tạo speaker embedding đi kèm model. Nếu `REFERENCE_FILE` rỗng, notebook lấy clip đầu tiên trong metadata dài 3–8 giây; có thể điền tên file để tự chọn. Đây là lựa chọn theo thời lượng, không phải đánh giá chất lượng. `--no-denoise` giữ audio theo cách bước chuẩn bị dataset xử lý.

Giọng fine-tune dùng **embedding**, không cần lưu mã audio tham chiếu. File `voices_v3_turbo.json` được ghi vào model và giọng này được đặt mặc định.


In [ ]:
if REFERENCE_FILE:
    reference = (DATASET_DIR / 'raw_audio' / REFERENCE_FILE).resolve()
    if not reference.is_relative_to((DATASET_DIR / 'raw_audio').resolve()):
        raise ValueError('REFERENCE_FILE phải nằm trong raw_audio.')
else:
    select_reference = """import csv, json, pathlib, soundfile as sf, sys
root = pathlib.Path(sys.argv[1])
with (root / 'metadata.csv').open(encoding='utf-8') as f:
    for row in csv.reader(f, delimiter='|'):
        if not row or row[0].strip().lower() == 'file_name':
            continue
        p = root / 'raw_audio' / row[0].strip()
        info = sf.info(str(p))
        if 3 <= info.duration <= 8:
            print(json.dumps(str(p))); break
    else:
        raise SystemExit('Không tìm thấy clip 3–8 giây; hãy điền REFERENCE_FILE.')
"""
    result = subprocess.check_output([str(PYTHON), '-c', select_reference, str(DATASET_DIR)], text=True)
    reference = Path(json.loads(result.strip().splitlines()[-1]))
if not reference.is_file():
    raise FileNotFoundError(reference)
print('Clip tạo embedding:', reference.name)
run_command([PYTHON, '-u', REPO_DIR / 'finetune/make_voice.py',
             '--audio', reference, '--name', VOICE_NAME, '--description', VOICE_DESCRIPTION,
             '--gender', VOICE_GENDER, '--base', BASE_MODEL, '--out', MERGED_DIR,
             '--default', '--no-denoise'], log_path=RUN_DIR / 'make_voice.log')


## 10. Nghe thử model đã fine-tune (tuỳ chọn)

Đổi `TEST_TEXT` ở cấu hình nếu muốn thử câu khác. Ô chạy SDK với `backend='pytorch'` và `device='cuda'` để chắc chắn dùng trọng số đã merge. Tiến trình kết thúc sẽ trả lại VRAM. Nếu chỉ cần tải model, đặt `RUN_DEMO=False` và tiếp tục xuất kết quả.


In [ ]:
DEMO_PATH = RUN_DIR / 'demo.wav'
if RUN_DEMO:
    demo_script = """import sys
from vieneu import Vieneu
model_dir, voice, text, output, subfolder, dtype = sys.argv[1:]
tts = Vieneu(mode='v3turbo', backbone_repo=model_dir, model_subfolder=subfolder,
             backend='pytorch', device='cuda', dtype=dtype)
audio = tts.infer(text, voice=voice)
tts.save(audio, output)
print('Đã ghi:', output)
"""
    run_command([PYTHON, '-u', '-c', demo_script, MERGED_DIR, VOICE_NAME, TEST_TEXT,
                 DEMO_PATH, MODEL_SUBFOLDER, 'bfloat16' if USE_BF16 else 'float32'],
                log_path=RUN_DIR / 'demo.log')
    from IPython.display import Audio, display
    display(Audio(filename=str(DEMO_PATH)))
else:
    print('Bỏ qua nghe thử theo RUN_DEMO=False.')


## 11. Xuất và tải kết quả

Ô này tạo ZIP riêng để tải nhanh adapter hoặc tải model đầy đủ. Không đóng gói cache, môi trường Python hay raw audio. `training_bundle.zip` chứa adapter cuối (nếu có), mọi checkpoint, cấu hình và log; `merged.zip` chứa model + giọng. `prepared_data.zip` chứa `train.parquet` nếu bật `EXPORT_PREPARED_DATA`.

- **Colab:** tải trong bảng Files bằng cách tìm `/content/vieneu_exports`, hoặc bật `DOWNLOAD_ON_COLAB=True`. Nếu train lưu trên Drive, các thư mục gốc vẫn có ở Drive; có thể chép ZIP sang Drive bằng ô tuỳ chọn bên dưới.
- **Kaggle:** tải từ `/kaggle/working/vieneu_exports` trong phần Output; lưu phiên bản notebook kèm outputs để giữ lại kết quả. Không cần chạy train lại chỉ để tạo ZIP nếu các file đã có trong phiên hiện tại.

Lưu cả `training_bundle.zip` và `merged.zip`: model dùng để suy luận, adapter/checkpoint dùng để merge lại hoặc so sánh. Parquet chứa dữ liệu đặc trưng được tạo từ audio/text của bạn.


In [ ]:
# Chuẩn hóa tên generic fast tokenizer để model xuất từ Transformers 5
# cũng nạp được bằng Transformers 4 (không đổi tokenizer.json hay token IDs).
for tokenizer_config in RUN_DIR.rglob('tokenizer_config.json'):
    saved_config = json.loads(tokenizer_config.read_text(encoding='utf-8'))
    if saved_config.get('tokenizer_class') == 'TokenizersBackend':
        saved_config['tokenizer_class'] = 'PreTrainedTokenizerFast'
        tokenizer_config.write_text(json.dumps(saved_config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'repo_url': REPO_URL, 'code_commit': CODE_COMMIT,
    'base_model': BASE_MODEL, 'model_subfolder': MODEL_SUBFOLDER,
    'selected_adapter': str(ADAPTER_DIR), 'voice_name': VOICE_NAME,
    'reference_file': reference.name, 'gpu': GPU_INFO, 'use_bf16': USE_BF16,
    'dataset_metadata_sha256': hashlib.sha256((DATASET_DIR / 'metadata.csv').read_bytes()).hexdigest(),
}
(RUN_DIR / 'notebook_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
frozen = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'environment.txt').write_text(frozen, encoding='utf-8')
if (WORK_ROOT / 'prepare.log').is_file():
    shutil.copy2(WORK_ROOT / 'prepare.log', RUN_DIR / 'prepare.log')

exports = []
training_zip = ARTIFACT_DIR / f'{RUN_NAME}_training_bundle.zip'
with zipfile.ZipFile(training_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in RUN_DIR.rglob('*'):
        relative = path.relative_to(RUN_DIR)
        if path.is_file() and relative.parts[0] != 'merged':
            archive.write(path, arcname=str(Path(RUN_NAME) / relative))
exports.append(training_zip)
if (MERGED_DIR / MODEL_SUBFOLDER / 'model.safetensors').is_file():
    merged_zip = Path(shutil.make_archive(str(ARTIFACT_DIR / f'{RUN_NAME}_merged'), 'zip',
                                         root_dir=RUN_DIR, base_dir='merged'))
    exports.append(merged_zip)
if EXPORT_PREPARED_DATA:
    prepared_zip = ARTIFACT_DIR / f'{RUN_NAME}_prepared_data.zip'
    with zipfile.ZipFile(prepared_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        archive.write(PARQUET_PATH, arcname='train.parquet')
    exports.append(prepared_zip)

from IPython.display import FileLink, display
for path in exports:
    print(f'{path} ({path.stat().st_size / 2**20:.1f} MB)')
    # Kaggle hiển thị đường dẫn /kaggle/working qua /files/ trong notebook.
    if IS_KAGGLE:
        relative = path.relative_to('/kaggle/working')
        display(FileLink(str(relative), url_prefix='/files/'))
if IS_COLAB and DOWNLOAD_ON_COLAB:
    from google.colab import files
    for path in exports:
        files.download(str(path))


### Tuỳ chọn: chép các ZIP sang Google Drive

Chỉ chạy khi muốn giữ ZIP trong Drive. Ô không làm gì nếu `COPY_ZIPS_TO_DRIVE=False`.


In [ ]:
COPY_ZIPS_TO_DRIVE = False
if COPY_ZIPS_TO_DRIVE:
    if not IS_COLAB:
        raise RuntimeError('Ô này chỉ dùng trên Colab.')
    from google.colab import drive
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/VieNeu/exports')
    destination.mkdir(parents=True, exist_ok=True)
    for path in exports:
        shutil.copy2(path, destination / path.name)
    print('Đã lưu ZIP tại:', destination)


## Dùng model sau khi tải về

Giải nén `*_merged.zip`; bạn sẽ có thư mục `merged/` chứa `update/model.safetensors` và `voices_v3_turbo.json`. Trong môi trường đã cài VieNeu và các phụ thuộc GPU tương thích:

```python
from vieneu import Vieneu

tts = Vieneu(
    mode='v3turbo',
    backbone_repo='/duong/dan/merged',
    backend='pytorch',
    device='cuda',
)
audio = tts.infer('Xin chào, đây là giọng đã fine-tune.', voice='Giong cua toi')
tts.save(audio, 'output.wav')
```

Đổi tên giọng cho khớp `VOICE_NAME`. Trên Windows có thể dùng đường dẫn như `r'D:\models\merged'`. Lần đầu suy luận vẫn cần Internet để tải codec MOSS và tài nguyên chưa có trong cache. File `environment.txt` ghi môi trường cloud đã dùng; đường dẫn editable trong đó cần thay bằng vị trí repo trên máy mới, không cài nguyên file này một cách máy móc.

## Khi gặp lỗi

| Hiện tượng | Cách xử lý |
|---|---|
| `Found an incompatible version of torchao` | Chạy lại ô cài đặt mới để gỡ torchao tùy chọn và kiểm tra LoRA. Giữ parquet, đổi RUN_NAME rồi train lại. |
| `ensurepip` thất bại khi tạo venv | Dùng ô cài đặt đã cập nhật (`--without-pip` và `pip --python`); chạy lại ô đó để sửa cả venv đã tạo dở. |
| Không có CUDA | Bật GPU trên nền tảng, kết nối runtime có GPU rồi chạy lại từ đầu. |
| Kaggle không tải được GitHub/Hugging Face | Bật Internet; nếu tuỳ chọn bị khoá, hoàn tất yêu cầu tài khoản của Kaggle hoặc dùng phiên cho phép Internet. |
| GPU không hỗ trợ BF16 | Notebook tự thêm `--no-bf16`. FP32 sẽ dùng nhiều VRAM hơn; giảm `BATCH_SIZE` nếu cần. |
| `CUDA out of memory` | Giảm `BATCH_SIZE` xuống 1 và tăng `GRAD_ACCUM` lên 16 để giữ batch hiệu dụng 16. Đổi `RUN_NAME` trước khi train lại. Chỉ giảm `MAX_LENGTH` khi hiểu rằng script sẽ bỏ những mẫu vượt giới hạn. |
| Train dừng/mất phiên | Lấy `checkpoint-*` đã lưu trên Drive hoặc outputs đã giữ; merge checkpoint đó. Chưa hỗ trợ resume optimizer trong script hiện tại. |
| Mất kết nối giao diện nhưng runtime còn chạy | Kết nối lại, xem `train_console.log` và output trước khi chạy lại ô train. |
| Run đã có kết quả | Ô train tự tạo tên lượt mới. Nếu muốn dùng kết quả cũ, bỏ qua ô train và merge adapter/checkpoint của lượt đó. |
| Tìm thấy nhiều dataset | Chọn số thứ tự bằng `DATASET_CHOICE` trong cấu hình rồi chạy lại cấu hình và ô nạp dataset; không cần nhập đường dẫn. |
| Parquet cũ sau khi sửa dataset | Đặt `REBUILD_PARQUET=True`, chạy lại chuẩn bị dữ liệu, dùng run mới để train. |
| Import thư viện lỗi sau khi đổi phiên bản | Dùng runtime mới, giữ revision cố định, chạy lại ô cài đặt và xem lỗi đầy đủ. `environment.txt` giúp đối chiếu các phiên bản đã dùng. |
| Số dòng train ít hơn metadata | Xem log prepare/train: pipeline bỏ clip ngoài 1–20 giây, phiên âm rỗng hoặc chuỗi vượt `MAX_LENGTH`. |
| Không tải được ZIP lớn qua trình duyệt | Colab: chép ZIP sang Drive. Kaggle: lưu notebook outputs rồi tải từ phần Output. |

Các lệnh được đối chiếu với `finetune/README.md`, `prepare_dataset.py`, `train_lora.py`, `merge_lora.py` và `make_voice.py` tại commit ghi ở ô cấu hình. Notebook chưa được chạy train trên GPU Colab/Kaggle trong môi trường soạn thảo này.
